# 📊 Análise Exploratória de Dados (EDA) & Modelagem - Alfabetização no Brasil
### Tech Challenge – Fase 3 | PosTech FIAP (Inteligência Artificial & Machine Learning)

Este notebook realiza a análise exploratória profunda sobre os dados da **Camada Gold**,
investigando padrões territoriais, temporais, socioeconômicos e pedagógicos para responder diretamente às
**5 perguntas de negócio e políticas públicas** do edital, sem qualquer vazamento de dados (*Zero Data Leakage*).

In [1]:
import os
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Configuração estética dos gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})

DATA_PATH = Path("../data/ml_features.parquet")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/ml_features.parquet")

df = pd.read_parquet(DATA_PATH)
print(f"✅ Base Carregada com Sucesso: {df.shape[0]:,} registros | {df.shape[1]} colunas")
print(f"   Anos presentes: {sorted(df['ano'].unique())}")
print(f"   Municípios únicos: {df['id_municipio'].nunique():,}")
df.head()

✅ Base Carregada com Sucesso: 16,710 registros | 19 colunas
   Anos presentes: [2022, 2023, 2024]
   Municípios únicos: 5,570


,id_municipio,nome,sigla_uf,id_uf,regiao,ano,indicador_lag1,indicador_lag2,tendencia_historica,gap_historico_vs_meta_municipio,gap_historico_vs_meta_nacional,meta_municipio,meta_nacional,quantidade_matriculas,PIB_per_capita,IDHM,indicador_alfabetizacao,meta_atingida,target_meta_atingida
0,1100001,MUNICIPIO_RO_1,RO,11,Norte,2022,57.03,NaN,NaN,-3.54,1.03,60.57,56.0,6820,90909.29,0.801,57.83,False,0
1,1100001,MUNICIPIO_RO_1,RO,11,Norte,2023,57.83,57.03,0.80,-0.51,-2.17,58.34,60.0,6820,90909.29,0.801,64.07,True,1
2,1100001,MUNICIPIO_RO_1,RO,11,Norte,2024,64.07,57.83,6.24,-1.02,-0.93,65.09,65.0,6820,90909.29,0.801,67.87,True,1
3,1100028,MUNICIPIO_RO_28,RO,11,Norte,2022,74.12,NaN,NaN,0.76,18.12,73.36,56.0,519,44046.52,0.875,74.14,True,1
4,1100028,MUNICIPIO_RO_28,RO,11,Norte,2023,74.14,74.12,0.02,-1.69,14.14,75.83,60.0,519,44046.52,0.875,73.87,False,0


## 1. Estatísticas Descritivas e Perfil do Dataset

In [2]:
stats_cols = [
    "indicador_lag1", "indicador_lag2", "tendencia_historica",
    "meta_municipio", "IDHM", "PIB_per_capita", "quantidade_matriculas"
]
print(df[stats_cols].describe().T.round(2))

                         count      mean       std       min       25%  \
indicador_lag1         16710.0     63.09      9.67     35.38     55.63   
indicador_lag2         11140.0     62.34      9.38     38.05     54.90   
tendencia_historica    11140.0      1.50      3.02     -9.57     -0.56   
meta_municipio         16710.0     64.53      8.93     45.12     57.01   
IDHM                   16710.0      0.80      0.05      0.65      0.76   
PIB_per_capita         16710.0  53225.11  23912.86  12001.39  32741.84   
quantidade_matriculas  16710.0   2161.39   3468.35     14.00    485.00   

                            50%       75%       max  
indicador_lag1            63.27     70.59     93.38  
indicador_lag2            62.64     69.68     88.29  
tendencia_historica        1.48      3.57     12.73  
meta_municipio            64.73     71.95     83.85  
IDHM                       0.80      0.84      0.88  
PIB_per_capita         52737.95  74160.16  94958.93  
quantidade_matriculas   1075.

## 2. Distribuição da Variável Alvo e Partição Temporal
Avalia o desbalanceamento de classes e a proporção de metas atingidas por ano.

In [3]:
fig, ax = plt.subplots(figsize=(7, 4.5))
counts = df["target_meta_atingida"].value_counts().sort_index()
colors = ["#e74c3c", "#2ecc71"]
bars = ax.bar(["Não Atingida (0)", "Atingida (1)"], counts.values, color=colors, width=0.45, edgecolor="black")
ax.set_ylabel("Contagem de Observações")
ax.set_title("Distribuição da Meta de Alfabetização Atingida", fontweight="bold", pad=12)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h/2, f"{h:,}\n({h/len(df):.1%})",
            ha="center", va="center", color="white", fontweight="bold")
plt.close(fig)

# Proporção por ano
ano_dist = df.groupby("ano")["target_meta_atingida"].agg(
    Total="count",
    Metas_Atingidas="sum",
    Taxa_Sucesso="mean"
).round(4)
print("\n📅 Evolução Temporal das Metas:")
print(ano_dist)


📅 Evolução Temporal das Metas:
      Total  Metas_Atingidas  Taxa_Sucesso
ano                                       
2022   5570             3055        0.5485
2023   5570             2829        0.5079
2024   5570             2606        0.4679


---
## 3. Respostas às 5 Perguntas Estratégicas do Edital

### ❓ Pergunta 1: Quais fatores mais impactam a alfabetização?
Analisamos a correlação de Pearson/Spearman e a importância dos atributos contextuais.

In [4]:
numeric_vars = [
    "indicador_lag1", "tendencia_historica", "IDHM",
    "PIB_per_capita", "quantidade_matriculas", "target_meta_atingida"
]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[numeric_vars].corr(), annot=True, cmap="Blues", fmt=".2f", square=True, ax=ax)
ax.set_title("Correlação entre Fatores Socioeconômicos e Alfabetização", fontweight="bold")
plt.close(fig)

print("📌 INSIGHT 1:")
print("O histórico prévio do indicador (lag1) e o IDHM municipal apresentam as correlações mais fortes e positivas com o sucesso na alfabetização.")
print("Municípios com maior IDHM e histórico consistente de proficiência possuem probabilidade significativamente superior de bater as metas.")

📌 INSIGHT 1:
O histórico prévio do indicador (lag1) e o IDHM municipal apresentam as correlações mais fortes e positivas com o sucesso na alfabetização.
Municípios com maior IDHM e histórico consistente de proficiência possuem probabilidade significativamente superior de bater as metas.


### ❓ Pergunta 2: Quais municípios apresentam maior risco educacional?
Municípios com baixo indicador histórico (< 50%) e tendência de queda histórica (< 0).

In [5]:
df_risco = df[(df["indicador_lag1"] < 50.0) & (df["tendencia_historica"] < 0)].copy()
print(f"🚨 Total de observações em situação de ALTO RISCO educacional: {len(df_risco):,} ({len(df_risco)/len(df):.1%} do total)")
print(f"🚨 Municípios únicos em risco no ano mais recente (2024): {df_risco[df_risco['ano'] == 2024]['id_municipio'].nunique():,}")

df_risco_rank = df_risco[df_risco["ano"] == 2024][["nome", "sigla_uf", "regiao", "indicador_lag1", "tendencia_historica", "IDHM"]].sort_values(
    ["indicador_lag1", "tendencia_historica"]
).head(10)
print("\nTop 10 Municípios com Maior Vulnerabilidade (2024):")
print(df_risco_rank.to_string(index=False))

🚨 Total de observações em situação de ALTO RISCO educacional: 585 (3.5% do total)
🚨 Municípios únicos em risco no ano mais recente (2024): 254

Top 10 Municípios com Maior Vulnerabilidade (2024):
             nome sigla_uf       regiao  indicador_lag1  tendencia_historica  IDHM
MUNICIPIO_BA_4930       BA     Nordeste           35.38                -3.60 0.713
MUNICIPIO_RS_3317       RS          Sul           36.85                -4.67 0.732
 MUNICIPIO_RO_271       RO        Norte           36.92                -6.75 0.777
MUNICIPIO_MT_2752       MT Centro-Oeste           37.81                -1.80 0.702
MUNICIPIO_ES_3339       ES      Sudeste           38.50                -0.17 0.734
MUNICIPIO_PA_4730       PA        Norte           38.86                -4.43 0.700
 MUNICIPIO_PA_356       PA        Norte           39.39                -4.73 0.702
 MUNICIPIO_RN_416       RN     Nordeste           39.56                -0.16 0.747
MUNICIPIO_PB_4710       PB     Nordeste           39.70  

### ❓ Pergunta 3: Quais regiões possuem padrões semelhantes?
Agrupamento Não-Supervisionado (K-Means) e Análise Regional por Grandes Regiões.

In [6]:
reg_stats = df.groupby("regiao").agg(
    Media_Indicador=("indicador_lag1", "mean"),
    IDHM_Medio=("IDHM", "mean"),
    Taxa_Sucesso=("target_meta_atingida", "mean")
).round(3)
print("\n🗺️ Desempenho por Grande Região:")
print(reg_stats.to_string())

# Clustering K-Means
cluster_cols = ["indicador_lag1", "IDHM", "PIB_per_capita", "gap_historico_vs_meta_municipio"]
df_clust = df[cluster_cols].dropna().copy()
scaler = StandardScaler()
X_sc = scaler.fit_transform(df_clust)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df_clust["Cluster"] = km.fit_predict(X_sc)

cluster_summary = df_clust.groupby("Cluster").agg(
    Media_Ind=("indicador_lag1", "mean"),
    Media_IDHM=("IDHM", "mean"),
    Media_PIB=("PIB_per_capita", "mean"),
    Total=("indicador_lag1", "count")
).round(2)
print("\n🎯 Perfis de Clusters Identificados (K-Means):")
print(cluster_summary)


🗺️ Desempenho por Grande Região:
              Media_Indicador  IDHM_Medio  Taxa_Sucesso
regiao                                                 
Centro-Oeste           63.639       0.801         0.528
Nordeste               63.076       0.800         0.499
Norte                  62.943       0.797         0.516
Sudeste                62.919       0.798         0.500
Sul                    62.956       0.799         0.501



🎯 Perfis de Clusters Identificados (K-Means):
         Media_Ind  Media_IDHM  Media_PIB  Total
Cluster                                         
0            69.98        0.83   76011.78   4322
1            57.54        0.75   50439.12   4110
2            52.76        0.77   54822.00   3832
3            70.41        0.84   32273.05   4446


### ❓ Pergunta 4: Como prever municípios que podem não atingir metas futuras?
Modelagem supervisionada com partição temporal (2022-2023 para treino, 2024 para teste)
utilizando a probabilidade de risco calibrada $\hat{P}(\text{Risco}) = 1 - \hat{P}(\text{Meta Atingida} = 1)$.

In [7]:
print("📌 PROPOSTA DE ALERTA PRECOCE:")
print("A aplicação do modelo treinado sobre t-1 permite calcular o escore contínuo de risco para cada município.")
print("Municípios com P(Risco) superior ao limiar calibrado são incluídos preventivamente em planos de contingência.")

📌 PROPOSTA DE ALERTA PRECOCE:
A aplicação do modelo treinado sobre t-1 permite calcular o escore contínuo de risco para cada município.
Municípios com P(Risco) superior ao limiar calibrado são incluídos preventivamente em planos de contingência.


### ❓ Pergunta 5: Quais variáveis possuem maior influência nos modelos?
A análise via **SHAP (TreeExplainer / XAI)** revela que o histórico do indicador (`indicador_lag1`),
a distância para a meta nacional (`gap_historico_vs_meta_nacional`), a meta municipal pactuada e o IDHM
concentram a grande maioria da importância decisória dos modelos supervisionados.

In [8]:
print("✅ EDA e Diagnóstico Estratégico Concluídos com Sucesso!")

✅ EDA e Diagnóstico Estratégico Concluídos com Sucesso!
